# Ropedia Academy — A3 · Temporal motion backbones

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A3.ipynb)

> **Three core ideas of motion backbones in one run: velocity (translation-invariant) features, a temporal-conv encoder, and ST-GCN skeleton-graph propagation.**
>
> 一次运行讲清运动骨干的三大要点：速度（平移不变）特征、时间卷积编码器，以及 ST-GCN 的骨架图传播。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A3

In [ ]:
import torch, torch.nn as nn

T, J = 32, 17                                   # 32 frames, 17 joints
seq = torch.randn(1, T, J, 3)                   # a motion clip (positions over time)

# Representation matters: VELOCITIES are translation-invariant (same action, anywhere).
shifted = seq + torch.tensor([5., 0., 0.])           # move the whole body in space
vel, vel2 = seq[:, 1:] - seq[:, :-1], shifted[:, 1:] - shifted[:, :-1]
print("velocity is translation-invariant:", torch.allclose(vel, vel2, atol=1e-5))

# (a) Temporal backbone: 1D conv over time encodes MOVEMENT, not a single pose.
x = seq.reshape(1, T, J*3).transpose(1, 2)      # (1, C=J*3, T)
tcn = nn.Sequential(nn.Conv1d(J*3, 128, 3, padding=1), nn.ReLU(),
                    nn.Conv1d(128, 128, 3, padding=1), nn.AdaptiveAvgPool1d(1))
print("temporal-conv motion feature:", tuple(tcn(x).squeeze(-1).shape))

# (b) ST-GCN idea: propagate along the SKELETON GRAPH (anatomical inductive bias).
A = torch.eye(J)
for i, j in [(0,1),(1,2),(2,3),(0,4),(4,5)]: A[i,j] = A[j,i] = 1   # toy kinematic edges
A = A / A.sum(1, keepdim=True)
graph_feat = torch.einsum('ij,btjc->btic', A, seq)   # mix connected joints
print("skeleton-graph smoothed motion:", tuple(graph_feat.shape))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A3
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks